In [ ]:
import cv2
import time
import sys
import os

#. 상위 2단계 폴더(프로젝트 루트 디렉토리)를 파이썬 모듈 검색 경로에 추가
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
from src.config import DEFAULT_VIDEO_PATH
from src.devices.camera import VideoStreamReader
from src.devices.buffer import FrameBufferManager

def main():
    print("🚀 Phase 1: 비디오 스트림 수집 및 버퍼링 테스트를 시작합니다...")
    
    # 1. 비디오 스트리머 오픈 및 헬스체크
    reader = VideoStreamReader(DEFAULT_VIDEO_PATH)
    if not reader.open() or not reader.check_health():
        print("❌ [경고] 비디오 파일 신호 점검 실패! 기기 이상 경고를 송출합니다.")
        return

    print(f"✅ [정상] 비디오 파일 연결 성공: {DEFAULT_VIDEO_PATH}")
    
    # 2. 프레임 버퍼 생성 (5프레임 중 3프레임 검출 조건)
    buffer_mgr = FrameBufferManager(buffer_size=5, min_detect_count=3)
    
    # 가벼운 OpenCV 얼굴 검출기 사용 (Phase 1 테스트용)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    frame_count = 0
    start_triggered = False

    while True:
        ret, frame = reader.read_frame()
        if not ret or frame is None:
            print("🎬 비디오 재생이 완료되었습니다.")
            break

        frame_count += 1
        
        # 회색조 변환 후 얼굴 검출
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        
        detected_list = [{"box": (x, y, w, h)} for (x, y, w, h) in faces]
        
        # 3. 버퍼에 결과 수집
        buffer_mgr.push_frame_result(detected_list)
        
        # 4. 검출 가동 조건 확인
        if buffer_mgr.is_start_condition_met():
            if not start_triggered:
                print(f"🎉 [시작 조건 충족!] 프레임 #{frame_count}: 최근 5개 프레임 중 3개 이상에서 얼굴 검출 성공! AI 분석 프로세스를 시작합니다.")
                start_triggered = True
        else:
            if start_triggered:
                print(f"⚠️ 프레임 #{frame_count}: 얼굴 미검출로 가동 대기 중...")
                start_triggered = False

    reader.release()
    print("✅ Phase 1 테스트가 성공적으로 종료되었습니다.")


NameError: name 'os' is not defined